# Prediction-time feature availability

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
REPORTS = ROOT / "reports"
assert (REPORTS / "run_manifest.json").exists(), "Run python -m fraudgraph.pipeline --download first"
manifest = json.loads((REPORTS / "run_manifest.json").read_text())
print("Evidence run:", manifest["run_id"])
def figure(name):
    display(Image(filename=str(REPORTS / "figures" / name)))


Evidence run: 20260923T163139886696Z


In [2]:
display(Markdown((REPORTS / "FEATURE_DICTIONARY.md").read_text()))
display(pd.read_csv(REPORTS / "graph_feature_summary.csv"))

# Feature dictionary

## Named transaction baseline (15 columns)

| Column(s) | Interpretation and availability |
|---|---|
| `total_BTC` | Source-provided transaction BTC amount; unscaled units from source |
| `fees` | Transaction fee in BTC |
| `size` | Transaction size in bytes |
| `num_input_addresses`, `num_output_addresses` | Address counts intrinsic to the transaction |
| `in_BTC_min/max/mean/median/total` | Five source-provided summaries of input BTC amounts |
| `out_BTC_min/max/mean/median/total` | Five source-provided summaries of output BTC amounts |

The slash notation expands to five real columns in each row. These values are assumed observable once the transaction is observed. Missing values use training medians, not class-specific or full-dataset values. Source definitions should not be interpreted more precisely than the released column names allow.

## Engineered graph context (8 columns)

All features use the directed observed subgraph induced by transactions whose time is at or before the scored bucket. Each transaction's features are frozen at that bucket's close. None use class labels.

| Column | Definition | Why it may help |
|---|---|---|
| `g_in_degree` | Number of observed predecessor transactions | Incoming fan-in pattern |
| `g_out_degree` | Number of observed successor transactions | Outgoing fan-out pattern |
| `g_total_degree` | Sum of directed in/out degree | Connectivity intensity; redundant by design for a linear model |
| `g_pagerank_scaled` | PageRank with damping 0.85, multiplied by snapshot node count | Relative flow centrality without a mechanically shrinking scale |
| `g_component_size` | Number of nodes in the weakly connected component | Whether activity sits in a small chain or larger structure |
| `g_component_density` | Directed edge count / [n(nâˆ’1)]; zero for singleton | Component connectivity adjusted for size |
| `g_neighbor_degree_mean` | Mean total directed degree of unique immediate neighbors; zero if none | Whether the transaction adjoins hubs |
| `g_reciprocal_fraction` | Number of neighbors connected in both directions / unique neighbors; zero if none | Reciprocal structure diagnostic; can be constant in transaction DAGs |

Constant/redundant features are reported rather than misrepresented as predictive signals. PageRank convergence is checked by NetworkX, and nonfinite feature values fail validation. No all-pairs paths, betweenness, or expensive community search is needed.

## Sensitivity and exclusions

`Local_feature_1` through `Local_feature_93` extend the named baseline in the sensitivity experiments. They are anonymized local features provided by the authors, not reverse-engineered financial quantities. `Aggregate_feature_1` through `Aggregate_feature_72` and supplied `in_txs_degree` / `out_txs_degree` are excluded from every experiment. `class`, `y`, `txId`, and `Time step` are never predictors. Exact per-experiment lists are saved in `feature_sets.json`.


,Unnamed: 0,min,max,mean,std,nunique
0,g_in_degree,0.000000,284.000000,1.150101,3.911132,145.0
1,g_out_degree,0.000000,472.000000,1.150101,1.894740,63.0
2,g_total_degree,1.000000,473.000000,2.300203,4.328377,145.0
3,g_pagerank_scaled,0.289092,90.749132,0.938307,1.669709,69551.0
4,g_component_size,1089.000000,7880.000000,4755.924537,1544.332842,49.0
5,g_component_density,0.000148,0.000986,0.000272,0.000106,49.0
6,g_neighbor_degree_mean,1.058824,473.000000,12.773144,28.705667,2931.0
7,g_reciprocal_fraction,0.000000,0.000000,0.000000,0.000000,1.0


In [3]:
from fraudgraph.graph import snapshot_features
nodes = pd.DataFrame({"txId":[1,2,3], "Time step":[1,1,2]})
edges = pd.DataFrame({"txId1":[1,2], "txId2":[2,3]})
a, g = snapshot_features(nodes, edges, 1)
b, _ = snapshot_features(nodes.iloc[:2], edges.iloc[:1], 1)
pd.testing.assert_frame_equal(a,b)
assert 3 not in g
display(a)

,txId,g_in_degree,g_out_degree,g_total_degree,g_pagerank_scaled,g_component_size,g_component_density,g_neighbor_degree_mean,g_reciprocal_fraction
0,1,0,1,1,0.701754,2,0.5,1.0,0.0
1,2,1,0,1,1.298246,2,0.5,1.0,0.0


This executable toy check confirms that an edge to a future node cannot change the earlier snapshot. Full tests also perturb labels. The same-step graph is allowed only under the declared bucket-close contract. Instantaneous fraud interception would need finer timestamps and a different replay.